In [1]:
import re
import numpy as np
import pandas as pd
import nltk

from scipy.sparse import vstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
nltk.download('stopwords')
stop_words = nltk.corpus.stopwords.words('portuguese')

def load_data(file_path):
    return pd.read_json(file_path, lines=True)


def create_x_y(df):
    x = df['text']
    y = df['label']
    return x, y

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
NON_PUN_LABEL = 0
PUN_LABEL = 1


def extract_pair_id(example_id):
    return re.sub(r"\.[HN]$", "", str(example_id))


def extract_pair_suffix(example_id):
    match = re.search(r"\.([HN])$", str(example_id))

    if match:
        return match.group(1)

    return None


def add_pair_columns(df):
    df = df.reset_index(drop=True).copy()
    df["label"] = df["label"].astype(int)
    df["pair_id"] = df["id"].apply(extract_pair_id)
    df["pair_suffix"] = df["id"].apply(extract_pair_suffix)
    return df


def validate_complete_pairs(df, split_name):
    pair_sizes = df.groupby("pair_id").size()
    invalid_size_pairs = pair_sizes[pair_sizes != 2]

    if len(invalid_size_pairs) > 0:
        raise ValueError(
            f"{split_name}: existem {len(invalid_size_pairs)} pares incompletos."
        )

    invalid_label_pairs = []

    for pair_id, group in df.groupby("pair_id"):
        labels = set(group["label"].tolist())

        if labels != {NON_PUN_LABEL, PUN_LABEL}:
            invalid_label_pairs.append(pair_id)

    if len(invalid_label_pairs) > 0:
        raise ValueError(
            f"{split_name}: existem {len(invalid_label_pairs)} pares sem uma label 0 e uma label 1."
        )

    print(f"{split_name}: pares válidos.")
    print(f"{split_name}: {len(pair_sizes)} pares, {len(df)} exemplos.")


def validate_suffix_label_consistency(df, split_name):
    invalid_rows = df[
        ((df["pair_suffix"] == "H") & (df["label"] != PUN_LABEL)) |
        ((df["pair_suffix"] == "N") & (df["label"] != NON_PUN_LABEL)) |
        (df["pair_suffix"].isna())
    ]

    if len(invalid_rows) > 0:
        raise ValueError(
            f"{split_name}: há exemplos em que .H/.N não corresponde à label esperada."
        )

    print(f"{split_name}: sufixo .H/.N consistente com as labels.")


def validate_no_pair_overlap(train_df, val_df, test_df):
    train_pairs = set(train_df["pair_id"])
    val_pairs = set(val_df["pair_id"])
    test_pairs = set(test_df["pair_id"])

    train_val_overlap = train_pairs.intersection(val_pairs)
    train_test_overlap = train_pairs.intersection(test_pairs)
    val_test_overlap = val_pairs.intersection(test_pairs)

    print("Train/validation pair overlap:", len(train_val_overlap))
    print("Train/test pair overlap:", len(train_test_overlap))
    print("Validation/test pair overlap:", len(val_test_overlap))

    if train_val_overlap or train_test_overlap or val_test_overlap:
        raise ValueError("Ainda existe vazamento de pair_id entre os splits.")

    print("OK: não há sobreposição de pares entre treino, validação e teste.")


def build_pair_index_dataframe(df, split_name):
    pair_rows = []

    for pair_id, group in df.groupby("pair_id"):
        if len(group) != 2:
            continue

        pun_rows = group[group["label"] == PUN_LABEL]
        non_pun_rows = group[group["label"] == NON_PUN_LABEL]

        if len(pun_rows) != 1 or len(non_pun_rows) != 1:
            continue

        pun_row = pun_rows.iloc[0]
        non_pun_row = non_pun_rows.iloc[0]

        pair_rows.append({
            "pair_id": pair_id,

            "pun_index": int(pun_row.name),
            "pun_id": pun_row["id"],
            "pun_text": pun_row["text"],

            "non_pun_index": int(non_pun_row.name),
            "non_pun_id": non_pun_row["id"],
            "non_pun_text": non_pun_row["text"]
        })

    pairs_df = pd.DataFrame(pair_rows)

    print(f"{split_name}: {len(pairs_df)} pares montados.")

    return pairs_df

In [4]:
df_train = add_pair_columns(load_data('corpus/train.jsonl'))
df_val = add_pair_columns(load_data('corpus/validation.jsonl'))
df_test = add_pair_columns(load_data('corpus/test.jsonl'))

validate_complete_pairs(df_train, "Train")
validate_complete_pairs(df_val, "Validation")
validate_complete_pairs(df_test, "Test")

validate_suffix_label_consistency(df_train, "Train")
validate_suffix_label_consistency(df_val, "Validation")
validate_suffix_label_consistency(df_test, "Test")

validate_no_pair_overlap(df_train, df_val, df_test)

train_pairs_df = build_pair_index_dataframe(df_train, "Train")
val_pairs_df = build_pair_index_dataframe(df_val, "Validation")
test_pairs_df = build_pair_index_dataframe(df_test, "Test")

x_train, y_train = create_x_y(df_train)
x_val, y_val = create_x_y(df_val)
x_test, y_test = create_x_y(df_test)

Train: pares válidos.
Train: 1995 pares, 3990 exemplos.
Validation: pares válidos.
Validation: 285 pares, 570 exemplos.
Test: pares válidos.
Test: 570 pares, 1140 exemplos.
Train: sufixo .H/.N consistente com as labels.
Validation: sufixo .H/.N consistente com as labels.
Test: sufixo .H/.N consistente com as labels.
Train/validation pair overlap: 0
Train/test pair overlap: 0
Validation/test pair overlap: 0
OK: não há sobreposição de pares entre treino, validação e teste.
Train: 1995 pares montados.
Validation: 285 pares montados.
Test: 570 pares montados.


In [5]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words)
x_train_vectorized = vectorizer.fit_transform(x_train)
x_val_vectorized = vectorizer.transform(x_val)
x_test_vectorized = vectorizer.transform(x_test)

In [6]:
def build_voting_model():
    rf_model = RandomForestClassifier(
        n_estimators=100,
        criterion='entropy',
        max_depth=15,
        random_state=40
    )

    lr_model = LogisticRegression(
        random_state=40,
        max_iter=100
    )

    svm_model = SVC(
        probability=True,
        random_state=40
    )

    voting_model = VotingClassifier(
        estimators=[
            ('rf', rf_model),
            ('lr', lr_model),
            ('svm', svm_model)
        ],
        voting='soft',
        n_jobs=30
    )

    return voting_model


voting_model = build_voting_model()

In [7]:
voting_model.fit(x_train_vectorized, y_train)

y_test_pred = voting_model.predict(x_test_vectorized)
y_test_proba = voting_model.predict_proba(x_test_vectorized)

print("=== MAIN TEST RESULTS: standard binary classification ===")
print(
    classification_report(
        y_test,
        y_test_pred,
        labels=[0, 1],
        target_names=["0", "1"],
        digits=4,
        zero_division=0
    )
)

main_cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1]
)

main_cm_df = pd.DataFrame(
    main_cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

print("=== MAIN CONFUSION MATRIX ===")
print(main_cm_df)

=== MAIN TEST RESULTS: standard binary classification ===
              precision    recall  f1-score   support

           0     0.4727    0.7000    0.5644       570
           1     0.4223    0.2193    0.2887       570

    accuracy                         0.4596      1140
   macro avg     0.4475    0.4596    0.4265      1140
weighted avg     0.4475    0.4596    0.4265      1140

=== MAIN CONFUSION MATRIX ===
        pred_0  pred_1
true_0     399     171
true_1     445     125


In [8]:
def compute_basic_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),

        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "precision_weighted": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "recall_weighted": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }


def compute_pair_metrics(predictions_df):
    pair_exact_results = []
    pair_ranking_results = []

    for pair_id, group in predictions_df.groupby("pair_id"):
        if len(group) != 2:
            continue

        pair_exact_correct = bool(
            (group["label"] == group["prediction"]).all()
        )

        pair_exact_results.append(pair_exact_correct)

        pun_rows = group[group["label"] == PUN_LABEL]
        non_pun_rows = group[group["label"] == NON_PUN_LABEL]

        if len(pun_rows) == 1 and len(non_pun_rows) == 1:
            pun_score = float(pun_rows.iloc[0]["score_pun"])
            non_pun_score = float(non_pun_rows.iloc[0]["score_pun"])

            pair_ranking_correct = pun_score > non_pun_score
            pair_ranking_results.append(pair_ranking_correct)

    return {
        "pair_exact_accuracy": float(np.mean(pair_exact_results)) if pair_exact_results else 0.0,
        "pair_ranking_accuracy": float(np.mean(pair_ranking_results)) if pair_ranking_results else 0.0,
        "evaluated_pairs": int(len(pair_exact_results))
    }


def apply_pairwise_decoding(predictions_df):
    decoded_df = predictions_df.copy()
    decoded_df["prediction_threshold"] = decoded_df["prediction"]

    for pair_id, group in decoded_df.groupby("pair_id"):
        if len(group) != 2:
            continue

        sorted_indices = group.sort_values(
            "score_pun",
            ascending=False
        ).index.tolist()

        pun_index = sorted_indices[0]
        non_pun_index = sorted_indices[1]

        decoded_df.loc[pun_index, "prediction"] = PUN_LABEL
        decoded_df.loc[non_pun_index, "prediction"] = NON_PUN_LABEL

    return decoded_df


test_predictions_df = df_test[["id", "pair_id", "text", "label"]].copy()
test_predictions_df["prediction"] = y_test_pred

# Score de trocadilho: probabilidade da classe 1 menos probabilidade da classe 0
test_predictions_df["score_pun"] = (
    y_test_proba[:, PUN_LABEL] - y_test_proba[:, NON_PUN_LABEL]
)

test_predictions_df["prob_0"] = y_test_proba[:, NON_PUN_LABEL]
test_predictions_df["prob_1"] = y_test_proba[:, PUN_LABEL]

main_pair_metrics = compute_pair_metrics(test_predictions_df)

print("\n=== AUXILIARY PAIR ANALYSIS: standard ensemble predictions ===")
print(f"Pair exact accuracy: {main_pair_metrics['pair_exact_accuracy']:.4f}")
print(f"Pair ranking accuracy: {main_pair_metrics['pair_ranking_accuracy']:.4f}")
print(f"Evaluated pairs: {main_pair_metrics['evaluated_pairs']}")


pairwise_test_predictions_df = apply_pairwise_decoding(test_predictions_df)

pairwise_metrics = compute_basic_metrics(
    pairwise_test_predictions_df["label"].tolist(),
    pairwise_test_predictions_df["prediction"].tolist()
)

pairwise_metrics.update(
    compute_pair_metrics(pairwise_test_predictions_df)
)

print("\n=== AUXILIARY ONLY: pairwise decoding with standard ensemble ===")
print(
    "Atenção: este resultado força cada par a ter uma previsão 1 e uma previsão 0. "
    "Use apenas como análise auxiliar, não como resultado principal."
)

for metric_name, metric_value in pairwise_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

print("\n=== AUXILIARY CLASSIFICATION REPORT: pairwise decoding ===")
print(
    classification_report(
        pairwise_test_predictions_df["label"],
        pairwise_test_predictions_df["prediction"],
        labels=[0, 1],
        target_names=["0", "1"],
        digits=4,
        zero_division=0
    )
)

pairwise_cm = confusion_matrix(
    pairwise_test_predictions_df["label"],
    pairwise_test_predictions_df["prediction"],
    labels=[0, 1]
)

pairwise_cm_df = pd.DataFrame(
    pairwise_cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

print("=== AUXILIARY CONFUSION MATRIX: pairwise decoding ===")
print(pairwise_cm_df)


=== AUXILIARY PAIR ANALYSIS: standard ensemble predictions ===
Pair exact accuracy: 0.0421
Pair ranking accuracy: 0.3070
Evaluated pairs: 570

=== AUXILIARY ONLY: pairwise decoding with standard ensemble ===
Atenção: este resultado força cada par a ter uma previsão 1 e uma previsão 0. Use apenas como análise auxiliar, não como resultado principal.
accuracy: 0.4228
precision_macro: 0.4228
recall_macro: 0.4228
f1_macro: 0.4228
precision_weighted: 0.4228
recall_weighted: 0.4228
f1_weighted: 0.4228
pair_exact_accuracy: 0.4228
pair_ranking_accuracy: 0.3070
evaluated_pairs: 570

=== AUXILIARY CLASSIFICATION REPORT: pairwise decoding ===
              precision    recall  f1-score   support

           0     0.4228    0.4228    0.4228       570
           1     0.4228    0.4228    0.4228       570

    accuracy                         0.4228      1140
   macro avg     0.4228    0.4228    0.4228      1140
weighted avg     0.4228    0.4228    0.4228      1140

=== AUXILIARY CONFUSION MATRIX: p